## Runtime requirement

This notebook assumes a Google Colab runtime with GPU.

Before connecting, go to:
Runtime → Change runtime type → T4 GPU (for example)

In [24]:
# @title GPU availability check
import torch
from IPython.display import display, HTML

# Show a visible warning message if GPU is not available
if not torch.cuda.is_available():
    display(HTML("""
    <div style="padding:12px; border-radius:8px; background-color:#fff3cd; color:#856404; border:1px solid #ffeeba;">
        <b>Warning:</b> GPU is not enabled.<br>
        Please go to <b>Runtime -> Change runtime type -> T4 GPU</b>.
    </div>
    """))
else:
    print("GPU is available:", torch.cuda.get_device_name(0))

GPU is available: Tesla T4


## Installation requirements

In [1]:
!pip install biopython fair-esm -q
!pip install freesasa -q

## Importing modules

In [2]:
import os, sys
import argparse
sys.path.append('./piaco2_public_revised/')

import json
import numpy as np
import torch

print("imports OK")

imports OK


In [17]:
# @title ESM-2 preloading
# ESM-2 loading
import esm, torch

model_name = "esm2_t33_650M_UR50D"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fn = getattr(esm.pretrained, model_name)
_model, _alphabet = fn()          # 1.5 GB
_model = _model.to(device).eval()
_batch_converter = _alphabet.get_batch_converter()

print(f"ESM-2 loaded on {device}  |  layers={_model.num_layers}")
del _model, _alphabet, _batch_converter

ESM-2 loaded on cuda  |  layers=33


In [14]:
# @title Data uploader
import os
import ipywidgets as widgets
from IPython.display import display, clear_output

# Directories for saving files
MAIN_DIR = "/content/pdb"
CHAIN_DIR = "/content/dmasif"

os.makedirs(MAIN_DIR, exist_ok=True)
os.makedirs(CHAIN_DIR, exist_ok=True)

# UI elements
title = widgets.HTML("<h3>PDB + dMaSIF Data Uploader</h3>")
desc = widgets.HTML("<p>Upload the main PDB file and the corresponding dMaSIF chain numpy files.</p>")

# Upload widget for the main PDB file
main_upload = widgets.FileUpload(
    accept=".pdb",
    multiple=False,
    description="Main PDB"
)

# Upload widget for the chain numpy files
chain_upload = widgets.FileUpload(
    accept=".npy",
    multiple=True,
    description="Chain npy"
)

status = widgets.Output()

ui = widgets.VBox([
    title,
    desc,
    widgets.HTML("<b>Step 1: Upload main PDB</b>"),
    main_upload,
    widgets.HTML("<b>Step 2: Upload chain numpy files (A and B)</b>"),
    chain_upload,
    status
])

# Store main pdb base name
main_base = {"name": None}

def handle_main_upload(change):
    """Handle upload of the main PDB file"""
    with status:
        clear_output()

        if not main_upload.value:
            print("No main PDB selected.")
            return

        item = list(main_upload.value.values())[0]
        filename = item["metadata"]["name"]
        content = item["content"]

        if not filename.lower().endswith(".pdb"):
            print("Error: Please upload a .pdb file.")
            return

        save_path = os.path.join(MAIN_DIR, filename)

        with open(save_path, "wb") as f:
            f.write(content)

        # Extract base name (xxxx)
        base = os.path.splitext(filename)[0]
        main_base["name"] = base

        print("✅ Main PDB uploaded")
        print("File:", filename)
        print("Saved to:", save_path)
        print("")
        print("Now upload:")
        print(f"{base}_chain_A.npy")
        print(f"{base}_chain_B.npy")


def handle_chain_upload(change):
    """Handle upload of chain numpy files"""
    with status:
        clear_output()

        if main_base["name"] is None:
            print("Please upload the main PDB first.")
            return

        base = main_base["name"]

        if not chain_upload.value:
            print("No chain files selected.")
            return

        required = {f"{base}_chain_A.npy", f"{base}_chain_B.npy"}
        uploaded = set()

        for item in chain_upload.value.values():
            filename = item["metadata"]["name"]
            content = item["content"]

            if not filename.endswith(".npy"):
                print(f"Skipping non-npy file: {filename}")
                continue

            save_path = os.path.join(CHAIN_DIR, filename)

            with open(save_path, "wb") as f:
                f.write(content)

            uploaded.add(filename)

        # Verify required files
        if required.issubset(uploaded):
            print("✅ Chain numpy files uploaded successfully")
            for f in required:
                print("Saved:", os.path.join(CHAIN_DIR, f))
        else:
            print("⚠ Missing required files.")
            print("Required:")
            for f in required:
                print("-", f)


# Attach handlers
main_upload.observe(handle_main_upload, names="value")
chain_upload.observe(handle_chain_upload, names="value")

display(ui)

## Configuration

In [18]:
PDB        = "1eej_1.pdb" # PDB file name
RECEPTOR   = "A"          # receptor chain ID
LIGAND     = "B"          # ligand chain ID
CHECKPOINT = "piaco2_public_revised/checkpoint/piaco2_yes_pl_yes_crattn/best_model.pth"
DEVICE     = "cuda:0"

# PDB
PDB_DIR        = "/content/pdb"
# dMaSIF
DMASIF_NPY_DIR = "/content/dmasif"

In [21]:
!python piaco2_public_revised/infer_pdb_pair.py \
    --pdb            {PDB_DIR}/{PDB} \
    --receptor       {RECEPTOR} \
    --ligand         {LIGAND} \
    --checkpoint     {CHECKPOINT} \
    --device         {DEVICE} \
    --dmasif_npy_dir {DMASIF_NPY_DIR} \
    --esm_pooling --esm_crossattn


[preprocess] /content/pdb/1eej_1.pdb
  Receptor chains: ['A']  Ligand chains: ['B']
  Raw atoms: 1988 receptor, 1988 ligand
  Interface residues: 41 receptor, 43 ligand
  Interface atoms:    362 receptor, 380 ligand
  Selected atoms: 362 receptor, 362 ligand
  Loaded dMaSIF: /content/dmasif/1eej_1_chain_A.npy  shape=(22861, 19)
  Loaded dMaSIF: /content/dmasif/1eej_1_chain_B.npy  shape=(22207, 19)
  After dMaSIF append: (1000, 52)
  Output shape: (1000, 52)
Total residues: 216 receptor, 216 ligand
Interface residues (distance-based): 41 receptor, 43 ligand
Calculating SASA changes...
SASA filtering: 41 -> 24 receptor residues
SASA filtering: 43 -> 22 ligand residues
[ESM interface] distance+SASA: 24 receptor, 22 ligand residues
{
  "pdb": "/content/pdb/1eej_1.pdb",
  "receptor": "A",
  "ligand": "B",
  "checkpoint": "/content/piaco2_public_revised/checkpoint/piaco2_yes_pl_yes_crattn/best_model.pth",
  "device": "cuda:0",
  "in_channels": 49,
  "npoint": 1000,
  "used_esm_pooling": tru